# 4.0 — fit HMMs to behavioural signals

**Gaussian** HMM for whisking, **Poisson** HMM for licking. Two states: high and low.

## There is no hyperparameter search

The AR-HMM needed a lag, and choosing it was the whole problem — held-out likelihood rises
monotonically with lag (no criterion identifies the order at this data size) while the
segmentation converges by lag ~16 and syllable duration drifts away from the model-free
changepoint duration. Dropping the autoregressive emission removes that axis: a Gaussian
HMM *is* the lag-0 limit, and "whisking is high or low" is what the downstream analysis
actually consumes.

Everything else was measured one knob at a time
(`hmm_diagnostics/gaussian_hyperparams.csv`), as frame agreement against the default fit:

| knob | perturbation | agreement |
|---|---|---|
| `transition_matrix_stickiness` (κ) | 1e3 | **1.0000** |
| κ | 1e5 | 0.9973 |
| `method` | `prior` instead of `kmeans` | **1.0000** |
| `emission_prior_scale` | ×1e2 | **1.0000** |
| `emission_prior_scale` | ×1e4 | **1.0000** |
| `emission_prior_concentration` | ×1e4 | **1.0000** |
| `transition_matrix_concentration` | 10 | **1.0000** |

Not "small" — bit-identical. So the priors and the initialisation are left at their
defaults and not searched.

**`num_states` is the exception, and it is deliberately not selected.** Held-out LL keeps
improving as states are added (−0.154 → +0.043 → +0.113 nats/frame for 2/3/4) while median
dwell collapses (567 → 200 → 117 ms), so CV would run away exactly as it did with lag.
`num_states = 2` is a stated modelling commitment, not a fitted quantity.

In [1]:
""" IMPORTS """
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"   # no GPU needed; silences the CUDA plugin probe
import numpy as np
import pandas as pd

import hmm_functions as H
from segmentation_functions import idxs_from_files

ERROR:2026-09-04 16:20:13,348:jax._src.xla_bridge:444: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/jax/_src/xla_bridge.py", line 442, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 324, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 281, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: CUDA_ERROR_NO_DEVICE


## Configuration

In [5]:
# ==========================================================================
#                                RUN CODE
# ==========================================================================

prefix = '/home/ines/repositories/representation_learning_variability/'

# ---- WHICH DATA -------------------------------------------------------------
data_path = prefix + 'paper-individuality/data/design_matrices/'
data_path = prefix + 'paper-individuality/data/design_matrices/1_camera_setup/NM/'  # session_1  last_training
# data_path = prefix + 'paper-individuality/data/design_matrices/1_camera_setup/extra_bwm/'

fps = 60.0        # 60 for the two-camera set, 30 for the 1_camera_setup / training sets
fps = 30.0        # 60 for the two-camera set, 30 for the 1_camera_setup / training sets

# ---- WHAT TO FIT ------------------------------------------------------------
# One entry per variable; the emission family travels with the variable.
#   zsc   z-score the signal (continuous levels yes, counts no)
#   model 'gaussian' | 'poisson' | 'bernoulli'
VARIABLES = [
    dict(var='whisker_me', model='gaussian', zsc=True,  method='kmeans'),
    dict(var='Lick count', model='poisson',  zsc=False, method='prior'),
    # dict(var='avg_wheel_vel', model='gaussian', zsc=True,  method='kmeans'),
    # 'bernoulli' binarises the counts. Better specified at 30 Hz (wins 9/10 on raw LL)
    # but NOT a drop-in: on a 60 Hz session it gave 67 ms dwell against Poisson's 1033 ms.
    # dict(var='Lick count', model='bernoulli', zsc=False, method='prior'),
]

# ---- FITTING PARAMETERS -----------------------------------------------------
num_states = 2            # a commitment: high vs low. See the note above.
num_train_batches = 5
fit_method = 'em'
num_iters = 100           # every fit reached 99.99% of its final log-prob by iteration 99
kappa = 0.0               # measured irrelevant; not searched
min_dwell_ms = 167.       # flicker screen, in ms so it means the same at 30 and 60 Hz

# ---- BIN COARSENESS ---------------------------------------------------------
# Fit on bins of BIN_MS instead of single frames. Specified in MILLISECONDS, not
# frames, so the same number means the same thing at 30 and 60 Hz.
#
#   None (or 0)  fit at the native frame rate -- the previous behaviour, bit-identical
#   100          fit on 100 ms bins  (3 frames at 30 Hz, 6 at 60 Hz)
#
# WHY: a single 33 ms frame holds 0 or 1 lick, so both Poisson states converge on the
# same near-zero rate (measured: 0.071 vs 0.071 licks/frame) and EM collapses -- one
# state swallowing the whole session. Measured on 6 collapsing sessions, EVERY width
# from 67 ms up fixes it; the emission rates separate as soon as a bin can hold 2 licks
# (0.000 vs 0.410 at 67 ms). Whisking does not need this: it is continuous and already
# bimodal, and 219/221 of its fits are fine.
#
# Counts are SUMMED across the frames in a bin, continuous levels AVERAGED (BIN_AGG in
# hmm_functions). The saved most_likely_states are always upsampled back to frame
# resolution, so nothing downstream changes -- the cost is that a state onset is
# quantised to the bin edge (~29 ms RMS at 100 ms, ~58 ms at 200 ms).
BIN_MS = {'whisker_me': None,      # continuous and bimodal: leave at the frame rate
          'Lick count': 100} #100.0
# a single number instead of a dict applies to every variable:
# BIN_MS = 100.0

def bin_frames_for(var):
    """BIN_MS -> whole frames. 1 means no binning."""
    ms = BIN_MS.get(var) if isinstance(BIN_MS, dict) else BIN_MS
    if not ms:
        return 1
    k = max(1, int(round(float(ms) * fps / 1000.0)))
    return k

n_jobs = 4                # ~3-4 GB per worker: 5-6 is the ceiling on a 31 GB machine

# ---- SESSION LIST -----------------------------------------------------------
all_files = os.listdir(data_path)
design_matrices = [item for item in all_files
                   if 'design_matrix' in item and 'standardized' not in item]

sessions_to_exclude = []      # paste an exclusion list here if you want one

filtered_design_matrices = np.array(
    [s for s in design_matrices if s.split('_')[2] not in sessions_to_exclude])
idxs, mouse_names = idxs_from_files(filtered_design_matrices)
print(f'{len(design_matrices)} design matrices -> {len(idxs)} sessions '
      f'(excluded {len(design_matrices) - len(filtered_design_matrices)})')

# ---- WHERE TO SAVE ----------------------------------------------------------
def paths_for(spec):
    """One directory per (variable, model), so fits never overwrite each other."""
    k = bin_frames_for(spec['var'])
    binpart = '' if k == 1 else f"_bin{k*1000.0/fps:.0f}ms"
    tag = f"{num_train_batches}_{spec['model']}_{spec['method']}_{fit_method}_" \
          f"zsc_{spec['zsc']}_k{int(kappa)}_s{num_states}{binpart}/"
    # save = prefix + 'paper-individuality/data/hmm/fits/' + tag
    save = prefix + 'paper-individuality/data/neuromodulators/hmm/fits/' + tag
    # states = prefix + 'paper-individuality/data/hmm/most_likely_states/' + tag
    states = prefix + 'paper-individuality/data/neuromodulators/hmm/most_likely_states/' + tag
    csv = os.path.join(save, f"assessments_{spec['var']}.csv")
    return save, states, csv

for spec in VARIABLES:
    k = bin_frames_for(spec['var'])
    width = 'native frames' if k == 1 else f"{k} frames = {k*1000.0/fps:.0f} ms"
    print(f"  {spec['var']:14s} {spec['model']:10s} {width:22s} -> {paths_for(spec)[0]}")

91 design matrices -> 91 sessions (excluded 0)
  whisker_me     gaussian   native frames          -> /home/ines/repositories/representation_learning_variability/paper-individuality/data/neuromodulators/hmm/fits/5_gaussian_kmeans_em_zsc_True_k0_s2/
  Lick count     poisson    3 frames = 100 ms      -> /home/ines/repositories/representation_learning_variability/paper-individuality/data/neuromodulators/hmm/fits/5_poisson_prior_em_zsc_False_k0_s2_bin100ms/


## Fit

Resumable: a session with complete output is skipped, so this can be interrupted and
rerun. The assessment CSV is rebuilt from the pickles each time, so it stays complete
across restarts.

In [6]:
results = {}
for spec in VARIABLES:
    save_path, states_save_path, csv_path = paths_for(spec)
    print(f"\n=== {spec['var']}  ({spec['model']}) ===")
    results[spec['var']] = H.run_all(
        idxs, [spec['var']], spec['model'], spec['zsc'], num_states, num_train_batches,
        spec['method'], fit_method,
        save_path=save_path, data_path=data_path, fps=fps,
        n_jobs=n_jobs, csv_path=csv_path, states_save_path=states_save_path,
        num_iters=num_iters, kappa=kappa, min_dwell_ms=min_dwell_ms,
        bin_frames=bin_frames_for(spec['var']))


=== whisker_me  (gaussian) ===
Found 0 sessions to process.
0 fitted this call; wrote 91 rows -> /home/ines/repositories/representation_learning_variability/paper-individuality/data/neuromodulators/hmm/fits/5_gaussian_kmeans_em_zsc_True_k0_s2/assessments_whisker_me.csv

=== Lick count  (poisson) ===
Found 91 sessions to process.


/home/ines/repositories/representation_learning_variability/paper-individuality/segmentation/hmm_functions.py:149: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[var].iloc[cut_idx:] = np.nan
/home/ines/repositories/representation_learning_

91 fitted this call; wrote 91 rows -> /home/ines/repositories/representation_learning_variability/paper-individuality/data/neuromodulators/hmm/fits/5_poisson_prior_em_zsc_False_k0_s2_bin100ms/assessments_Lick count.csv


## Review

Three **independent** failure modes — held-out likelihood does not predict a degenerate
segmentation, so each has to be checked separately. Use `4.1_hmm_inspect.ipynb` to look at
the fits themselves.

In [13]:
for spec in VARIABLES:
    save_path, _, csv_path = paths_for(spec)
    if not os.path.exists(csv_path):
        print(f"{spec['var']}: nothing fitted yet")
        continue
    df = pd.read_csv(csv_path)
    print(f"\n=== {spec['var']}  ({spec['model']})  —  {len(df)} sessions ===")
    for flag in ['collapsed', 'degenerate_occupancy', 'flickering']:
        if flag in df:
            print(f'  {flag:22s} {int(df[flag].fillna(False).sum()):4d}')
    print(f"  {'errors':22s} {int((df.error.fillna('') != '').sum()):4d}")
    print(f"  {'fit_ok':22s} {int(df.fit_ok.sum()):4d} / {len(df)}")
    print(f"  median dwell (ms)      {df.median_dwell_ms.median():7.1f}")
    print(f"  median occupancy       {df.occupancy_state1.median():7.3f}")
    if 'bin_ms' in df:
        print(f"  {'bin width (ms)':22s} {df.bin_ms.median():7.1f}")
    cols = [c for c in ['mouse','eid','bin_ms','median_dwell_ms','n_segments',
                        'occupancy_state1','bits_LL','level_low','level_high','error']
            if c in df]
    bad = df[~df.fit_ok]
    if len(bad):
        print('  flagged:')
        display(bad[cols])


=== whisker_me  (gaussian)  —  223 sessions ===
  collapsed                 0
  degenerate_occupancy      1
  flickering                1
  errors                    2
  fit_ok                  219 / 223
  median dwell (ms)        633.3
  median occupancy         0.585


/tmp/ipykernel_63915/2987116395.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  print(f'  {flag:22s} {int(df[flag].fillna(False).sum()):4d}')


TypeError: bad operand type for unary ~: 'float'